[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Why Documents


## What you will be able to do

Say what a document database stores and how that differs from five tables and four joins, with one
order written both ways in front of you. Start a MongoDB inside this notebook, make it a single node
replica set, and seed it, which is the cell every other notebook in this guide opens with. Read that
cell line by line and say what each part is for. And recognize the three ways it fails, all of which
hide the real message somewhere you would not look.


## The idea

### The problem

An order is one thing. It has a customer, some lines, a few tags. In a relational database that is
four or five tables, a foreign key in each, a join to read it back, and a migration to add a field.
None of that structure is in the order. It is there because rows are flat and the order is not.

A document database stores the order as the order. The nesting survives the round trip, and the
shape is declared nowhere.

### What a document is

A BSON document, which you write as a Python `dict`: strings, numbers, dates, lists, and other
documents, nested as deeply as you like. A collection is a lot of them, and it has no schema unless
you add one.

### Why it works that way

MongoDB indexes and queries inside documents, so `lines.sku` is a field you can index and match on
even though `lines` is an array of subdocuments. The join disappears because the thing you would
have joined is already there.

That is not free, and this guide is largely about the cost: **Modeling Without Joins** is where the
embedded array stops fitting, and **Bulk Writes and Transactions** is where one write covering two
documents turns out to be two writes.

### Where this shows up

Anything whose natural unit is an object rather than a row: orders, events, documents, product
catalogs with different fields per kind of product. Also anywhere the shape changes often enough
that a migration per change is the bottleneck.

### What this notebook covers

The same order relationally and as a document. The boot cell: install, server, replica set, seed,
each line explained. Then the three failures, which are all about `--fork` putting the reason in a
log file and printing something useless.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import pymongo

client = pymongo.MongoClient("mongodb://127.0.0.1:27017/shop", tz_aware=True)
shop = client.get_default_database()

order = {                                          # one object, stored as one object
    "reference": "AB-1099",
    "customer": {"name": "R. Okafor", "email": "r@example.com"},
    "lines": [
        {"sku": "LAP-000000", "quantity": 1, "price": 1689.62},
        {"sku": "MOU-000003", "quantity": 2, "price": 24.50},
    ],
    "tags": ["priority", "gift"],
}

shop.orders.delete_many({"reference": "AB-1099"})
shop.orders.insert_one(order)

found = shop.orders.find_one({"reference": "AB-1099"})
print("lines:", len(found["lines"]), "| second sku:", found["lines"][1]["sku"])
print("customer:", found["customer"]["name"])
print("came back the same shape it went in:", found["lines"] == order["lines"])
print("no join, no foreign key, and no table was declared first")
client.close()
```

```
lines: 2 | second sku: MOU-000003
customer: R. Okafor
came back the same shape it went in: True
no join, no foreign key, and no table was declared first
```

One `insert_one`, one `find_one`, and the nested list of lines came back as a nested list of lines.
Nothing created a table, nothing declared a column, and nothing joined anything.


## Setup

Seven imports, MongoDB, and the boot cell.

- `pymongo` is the driver, and the install pins it with `beanie` so both halves of the guide agree
- `subprocess` and `os` install and start the server, `sys` names this Python, `time` waits for it
- `random` seeds the data the same way every run, with `version` and `PackageNotFoundError`

This is the longest Setup in the collection and it is the same in all sixteen notebooks, so it is
worth reading once here. `start_server` installs MongoDB if there is none and starts it with a
replica set name. `initiate` turns that single node into a replica set, which is what
**Bulk Writes and Transactions** and **Migrations** need. `seed` fills `shop.products` and
`shop.reviews` from `random.seed(0)`, so every run of every notebook sees identical data.

If you already have a MongoDB running on `127.0.0.1:27017`, the cell finds it and leaves it alone.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import pymongo

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


## Worked examples

### The same order, both ways

Relationally, an order of two lines is rows in several tables, and reading it back is a join:


In [2]:
orders = [{"id": 1, "reference": "AB-1099", "customer_id": 7}]
customers = [{"id": 7, "name": "R. Okafor", "email": "r@example.com"}]
lines = [{"id": 1, "order_id": 1, "sku": "LAP-000000", "quantity": 1, "price": 1689.62},
         {"id": 2, "order_id": 1, "sku": "MOU-000003", "quantity": 2, "price": 24.50}]
tags = [{"order_id": 1, "tag": "priority"}, {"order_id": 1, "tag": "gift"}]

print("four tables:", len(orders), len(customers), len(lines), len(tags), "rows")
print("to read one order you join all four on ids that exist only to make the join possible")


four tables: 1 1 2 2 rows
to read one order you join all four on ids that exist only to make the join possible


The `customer_id`, the `order_id` on every line, and the whole `tags` table exist because a row
cannot hold a list. They are not facts about the order.

As a document, the order is the order:


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

order = {
    "reference": "AB-1099",
    "customer": {"name": "R. Okafor", "email": "r@example.com"},
    "lines": [{"sku": "LAP-000000", "quantity": 1, "price": 1689.62},
              {"sku": "MOU-000003", "quantity": 2, "price": 24.50}],
    "tags": ["priority", "gift"],
}

shop.orders.delete_many({"reference": "AB-1099"})                   # so the cell can be rerun
shop.orders.insert_one(order)

found = shop.orders.find_one({"reference": "AB-1099"}, {"_id": 0})  # leave the _id out for now
print("what came back:", found["customer"]["name"], "|", len(found["lines"]), "lines |",
      found["tags"])
print("identical to what went in:", found == {k: v for k, v in order.items() if k != "_id"})


what came back: R. Okafor | 2 lines | ['priority', 'gift']
identical to what went in: True


### Querying inside the document

The nesting is not opaque. MongoDB matches on paths through it, with dots:


In [4]:
print("by a nested field:   ",
      shop.orders.count_documents({"customer.name": "R. Okafor"}))
print("by a field inside an array element:",
      shop.orders.count_documents({"lines.sku": "MOU-000003"}))
print("by a value in an array:",
      shop.orders.count_documents({"tags": "gift"}))


by a nested field:    1
by a field inside an array element: 1
by a value in an array: 1


The third one is worth a second look. `{"tags": "gift"}` matches a document whose `tags` is the
string `"gift"` **or** whose `tags` is an array containing it. MongoDB treats a query against an
array as a query against each of its elements, which is why no `$contains` operator exists.

**Query Operators** is where that convenience turns into the guide's most common quiet failure.

### The seeded collection

Every notebook has these, from the cell above:


In [5]:
print("products:", shop.products.count_documents({}))
print("reviews: ", shop.reviews.count_documents({}))
print()
print("one product:")
for field, value in sorted(shop.products.find_one({"_id": 0}).items()):
    print(f"  {field:8} {value}")


products: 500
reviews:  767

one product:
  _id      0
  kind     laptop
  maker    Aster
  name     Aster laptop 0
  price    1689.62
  size     {'w': 21, 'h': 34}
  sku      LAP-000000
  stock    388
  tags     ['bulk', 'sale']


`_id` is the one field every document has. Here it was chosen, which is why it is a small integer
and why this notebook can print it; left out, MongoDB invents an `ObjectId`, which is twelve bytes
of mostly randomness and different on every run. **Collections and Documents** is about that.

### When to reach for which

| What you want | A document store | A relational database |
|---|---|---|
| one object with nested parts | one document, read with `find_one` | several tables and a join |
| a list inside a record | an array field | a second table |
| a field only some records have | leave it out of the rest | a nullable column, or another table |
| to change the shape | write the new shape | a migration, before any code runs |
| a guarantee every record has a field | a `$jsonSchema` validator, added by you | `NOT NULL`, there by default |
| one write covering several objects | a transaction, and a replica set | a transaction, always available |
| to count by a field across everything | an aggregation pipeline | `GROUP BY` |

The default is the relational one. Reach for documents when the unit of work really is an object
that is read and written whole, and be honest that the second half of that table is the price. This
guide's job is to make the price visible rather than to sell the idea.

### One order, written and read the way the rest of the guide will, finished


In [6]:
def place_order(shop, reference, customer, lines, tags=()):
    """One document, one round trip, and the total computed from what is in it."""
    order = {
        "reference": reference,
        "customer": customer,
        "lines": list(lines),
        "tags": list(tags),
        "total": round(sum(line["quantity"] * line["price"] for line in lines), 2),
    }
    shop.orders.replace_one({"reference": reference}, order, upsert=True)   # rerunnable
    return order["total"]


def read_order(shop, reference):
    """Everything about it, in one query, with no join anywhere."""
    return shop.orders.find_one({"reference": reference}, {"_id": 0})


total = place_order(shop, "AB-1100",
                    {"name": "T. Halloran", "email": "t@example.com"},
                    [{"sku": "MON-000001", "quantity": 2, "price": 310.00},
                     {"sku": "CAB-000004", "quantity": 5, "price": 9.99}],
                    tags=["bulk"])

back = read_order(shop, "AB-1100")
print("total:", total)
print("read back:", back["customer"]["name"], "|", len(back["lines"]), "lines |", back["tags"])
print("the same order, found by a field inside its lines:",
      shop.orders.count_documents({"lines.sku": "CAB-000004"}))


total: 669.95
read back: T. Halloran | 2 lines | ['bulk']
the same order, found by a field inside its lines: 1


`replace_one(..., upsert=True)` is there so the cell can be run twice, and it is also the first
appearance of a trap: it replaces the whole document, so a field not in `order` is gone.
**Update Operators** is where that costs somebody an afternoon.

### Where each part came from

| In the function | What it relies on | The section that showed it |
|---|---|---|
| a nested `customer` | a document holding a document | The same order, both ways |
| `lines` as a list of documents | an array of subdocuments | The same order, both ways |
| `{"lines.sku": ...}` | dot notation into an array | Querying inside the document |
| `{"_id": 0}` | a projection leaving `_id` out | The seeded collection |
| `shop` at all | the boot cell, and `get_default_database` | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/01-why-documents-solutions.ipynb).

**1.** Insert one document with a nested field and an array, and read it back.


In [7]:
# your code here


**2.** Count the seeded products, and print one of them.


In [8]:
# your code here


**3.** Find every product whose `size.w` is more than fifty, using dot notation.


In [9]:
# your code here


**4.** Find every product tagged `sale`, without any operator.


In [10]:
# your code here


**5.** Ask the server whether it is a replica set, and what it calls itself.


In [11]:
# your code here


**6.** Store an order whose lines have different fields from each other, and read it back.


In [12]:
# your code here


## Common errors

### DBPathInUse: Another mongod instance is already running


In [13]:
code, out = shell(f"mongod --dbpath {DBPATH} --port 27099 --fork --logpath {DBPATH}-second.log")

print("what the cell printed:")
print("  exit code:", code)
print(" ", out.splitlines()[-2] if len(out.splitlines()) > 1 else out)


what the cell printed:
  exit code: 1
  ERROR: child process failed, exited with 1


That is everything `--fork` will tell you: a number and a sentence with no cause in it. The reason
is in the log file, which is where every startup failure in this notebook goes:


In [14]:
for line in shell(f"tail -40 {DBPATH}-second.log")[1].splitlines():
    if "DBPathInUse" in line:
        print(line.split('"error":"')[1].rstrip('"}'))


DBPathInUse: Unable to lock the lock file: /tmp/guide_mongo/rs/mongod.lock (Resource temporarily unavailable). Another mongod instance is already running on the /tmp/guide_mongo/rs directory


The lock is taken during storage startup, before anything binds a port, so this is not
`Address already in use` and changing the port does not help. One mongod per data directory, and
the boot cell checks whether one is answering before it tries.

### NonExistentPath: Data directory not found


In [15]:
code, out = shell(f"mongod --dbpath {DBPATH}-missing --port 27099 "
                  f"--fork --logpath {DBPATH}-third.log")
print("exit code:", code, "and again the reason is in the log:")

for line in shell(f"tail -40 {DBPATH}-third.log")[1].splitlines():
    if "NonExistentPath" in line:
        print(" ", line.split('"error":"')[1].rstrip('"}').split(" Create the")[0])


exit code: 1 and again the reason is in the log:
  NonExistentPath: Data directory /tmp/guide_mongo/rs-missing not found.


mongod will not create its own data directory. The boot cell calls `os.makedirs(DBPATH,
exist_ok=True)` first, which is one line and removes the whole failure.

### pymongo.errors.ServerSelectionTimeoutError


In [16]:
start = time.perf_counter()
try:
    with pymongo.MongoClient("mongodb://127.0.0.1:27099/?directConnection=true",
                             serverSelectionTimeoutMS=3000) as nothing_there:
        nothing_there.admin.command("ping")
except pymongo.errors.ServerSelectionTimeoutError as error:
    detail = str(error).split(" (configured")[0]                    # the rest is timeout settings
    address, _, reason = detail.partition("] ")                     # the errno differs by platform
    print("waited", f"{time.perf_counter() - start:.0f}s", "then:", type(error).__name__)
    print("  it says:", reason, "at", address.split(" ")[0].rstrip(":"))


waited 3s then: ServerSelectionTimeoutError
  it says: Connection refused at 127.0.0.1:27099


Note what it did **not** do: fail immediately. PyMongo waits for a server to become selectable, and
the default wait is thirty seconds, so a query sent before the server is up looks like a hang rather
than an error. The cell above set `serverSelectionTimeoutMS=3000` to keep it short.

`directConnection=true` matters too. Without it, a client asks the server about its replica set
before deciding it is usable, and a mongod started with `--replSet` but never initiated reports
itself as a ghost that nothing can be sent to. That is why `initiate` uses its own direct client.

### No error: sudo systemctl start mongod


In [17]:
code, out = shell("systemctl start mongod")
first = out.splitlines()[0] if out else "(nothing)"

print("exit code:", code)
if code == 127:
    print("  systemctl is not installed at all here")
elif "not been booted with systemd" in out:
    print("  systemd is installed but is not running, which is the Colab case")
else:
    print(" ", first)
print()
print("either way there is no service to start, which is why the boot cell runs mongod itself")


exit code: 127
  systemctl is not installed at all here

either way there is no service to start, which is why the boot cell runs mongod itself


Every MongoDB installation guide tells you to start the service with `systemctl`. That instruction
assumes an init system, and a container usually has none: on Colab `systemctl` exists and reports
that the system was never booted with systemd, and on a Mac the command is not there at all. The
cell prints whichever case it got, because the failure looks different depending on where you run
it and both look like something you did wrong.

Running `mongod` yourself, with an explicit `--dbpath` and `--logpath`, works in all three places,
and that is why the boot cell looks the way it does.


In [18]:
client.close()
print("client closed")


client closed


## Recap

- A document is a BSON object, written as a Python `dict`, nested as deeply as you like. A
  collection is a lot of them and has no schema until you add one.
- An order that is four tables and a join relationally is one document and one `find_one` here, and
  it comes back the shape it went in.
- Dot notation queries inside a document, including into arrays: `customer.name`, `lines.sku`.
- A query against an array field matches if any element matches, which is why there is no
  `$contains`.
- The boot cell installs MongoDB, starts `mongod` with `--replSet rs0`, initiates the single node so
  transactions and migrations work, and seeds from `random.seed(0)`. It is the same in all sixteen
  notebooks and it is idempotent.
- `--fork` prints only an exit code. Every startup failure's real message is in the log file, and
  the two common ones are `DBPathInUse` and `NonExistentPath`.
- A query before the server is up is a thirty second wait, not an immediate error, unless you set
  `serverSelectionTimeoutMS`.


## What is next

**Collections and Documents** is the writing half: `insert_one` and `insert_many`, the `_id` you did
not choose and cannot print twice, and the collection that a typo creates for you with no error at
all.


---

[PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Collections and Documents](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/02-collections-and-documents.ipynb) &#8594;
